# Occupation Signals Before Polity Collapse: A Difference-in-Differences Approach

The relative prevalence of specific professional groups within a society can serve as an indicator of institutional development and, in some cases, as a precursor to periods of social or political disruption (Turchin, 2018). The Cultura database enables the computation of the proportion of individuals recorded under a given occupation within a polity or region over time.

**Methodology:**

1. We divide terminated polities into **short-lived** (< 150 years) and **long-lasting** (≥ 150 years). For short-lived polities, we compare occupation shares **25 years before** and **at the time of collapse**; for long-lasting polities, **100 years before** and **at the time of collapse**.
2. We require at least **100 recorded individuals** per polity and at least **10 individuals with a specific occupation** to avoid noise. We use the polity's end date as the collapse date and exclude all modern polities.
3. We apply a **Wilcoxon signed-rank test** to detect systematic changes in occupation share across polities.
4. To isolate collapse-specific signals from broader secular trends, we apply a **difference-in-differences** design, comparing each collapsed polity against its **5 geographically closest peers**.
5. We plot the polities with the largest occupation-share changes alongside their top 3 regional neighbours.

In [1]:
# === notebook config (auto-managed; edit values, not the tag) ===
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Database
DB_PATH = "../data/humans_clean.sqlite3"

# Figure style — minimal, Nature/Science publication standard
FIGSIZE = (8, 5)
DPI = 120
FONT_TITLE = 16
FONT_LABEL = 13
FONT_TICK = 11
FONT_LEGEND = 10

# Light, restrained palette (avoid AI-slop saturation)
COLOR_PRIMARY = "#2171b5"
COLOR_SECONDARY = "#b5542a"
COLOR_NEUTRAL = "#7f7f7f"
COLOR_LIGHT = "#d9d9d9"
COLOR_ACCENT = "#6a9e3a"
PALETTE = [COLOR_PRIMARY, COLOR_SECONDARY, COLOR_ACCENT, COLOR_NEUTRAL, COLOR_LIGHT]

import matplotlib as _mpl
_mpl.rcParams.update({
    "figure.figsize": FIGSIZE,
    "figure.dpi": DPI,
    "axes.titlesize": FONT_TITLE,
    "axes.labelsize": FONT_LABEL,
    "xtick.labelsize": FONT_TICK,
    "ytick.labelsize": FONT_TICK,
    "legend.fontsize": FONT_LEGEND,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})


## 1. Data Loading

We load all terminated (non-modern) polities from `cliopatria_polity_periods`, classify them by duration, and collect occupation shares in two comparison windows per polity type:
- **Short-lived (< 150 yr):** crisis window [t−25, t] vs. pre-crisis window [t−50, t−25]
- **Long-lasting (≥ 150 yr):** crisis window [t−100, t] vs. pre-crisis window [t−200, t−100]

We also compute polity centroids (mean lat/lon of matched individuals) to identify the 5 nearest geographic neighbours for the difference-in-differences control.

In [2]:
import sqlite3, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter, defaultdict
from scipy.stats import wilcoxon
from scipy.spatial.distance import cdist

DB = DB_PATH
conn = sqlite3.connect(DB)
conn.execute('PRAGMA cache_size=-500000')


In [3]:
# !pip install pyarrow fastparquet

In [4]:
# ── Parameters ──
DURATION_THRESHOLD = 150   # years: long-lived if >= 150
MIN_DURATION       = 26    # minimum polity duration (exclude < 26 years)
MIN_INDIVIDUALS    = 100   # minimum individuals per polity
MODERN_CUTOFF      = 2000  # exclude polities ending after this year

# Time windows (decade-based)
CRISIS_WINDOW      = 10    # crisis = last decade before collapse
PRE_CRISIS_GAP     = 20    # pre-crisis starts 20 years before collapse (2 decades before crisis)
PRE_CRISIS_WINDOW  = 10    # pre-crisis = 1 decade window

import os
from tqdm import tqdm

CHECKPOINT_DIR = '../checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(name, df):
    df.to_pickle(f'{CHECKPOINT_DIR}/{name}.pkl')
    print(f"  ✓ Saved: {name}.pkl")

def load_checkpoint(name):
    path = f'{CHECKPOINT_DIR}/{name}.pkl'
    if os.path.exists(path):
        print(f"  ✓ Loaded: {name}.pkl")
        return pd.read_pickle(path)
    return None

def extract_lowest_level_polity(polity_chain):
    """Extract the lowest-level polity from a hierarchy chain."""
    segments = polity_chain.split(';')
    for seg in reversed(segments):
        seg = seg.strip()
        if not seg.startswith('(') and not seg.endswith(')'):
            return seg
    return None

def get_lowest_level_polity_id(polity_chain, polity_id_chain):
    """Get the polity_id corresponding to the lowest-level polity."""
    segments = polity_chain.split(';')
    ids = polity_id_chain.split(';')
    for i in range(len(segments) - 1, -1, -1):
        seg = segments[i].strip()
        if not seg.startswith('(') and not seg.endswith(')'):
            return int(ids[i]) if i < len(ids) else int(ids[-1])
    return int(ids[-1])

def get_largest_polygon_centroid(geom):
    """Extract centroid from the largest polygon in a geometry."""
    gtype = geom.get('type', '')
    if gtype == 'Polygon':
        coords = geom['coordinates'][0]
        return np.mean([c[1] for c in coords]), np.mean([c[0] for c in coords])
    elif gtype == 'MultiPolygon':
        largest_poly = max(geom['coordinates'], key=lambda p: len(p[0]))[0]
        return np.mean([c[1] for c in largest_poly]), np.mean([c[0] for c in largest_poly])
    return None, None

# ── 1a. Load raw data ──
df_raw = load_checkpoint('df_raw')
if df_raw is None:
    print("Loading data from database...")
    df_raw = pd.read_sql_query('''
        SELECT wikidata_id, polity_name, polity_id, floruit_year
        FROM consolidated_database
        WHERE floruit_year IS NOT NULL AND occupations IS NOT NULL
    ''', conn)
    print(f"Loaded {len(df_raw):,} individuals")

    print("Extracting lowest-level polities...")
    df_raw['lowest_polity'] = df_raw['polity_name'].apply(extract_lowest_level_polity)
    df_raw['lowest_pid'] = df_raw.apply(lambda r: get_lowest_level_polity_id(r['polity_name'], r['polity_id']), axis=1)
    df_raw = df_raw[df_raw['lowest_polity'].notna()].copy()
    
    save_checkpoint('df_raw', df_raw)

print(f'Individuals: {len(df_raw):,}')

# ── 1b. Get polity metadata ──
df_polity_meta = load_checkpoint('df_polity_meta')
if df_polity_meta is None:
    print("Building polity metadata...")
    
    # Aggregate by lowest_polity
    lowest_level_counts = df_raw.groupby('lowest_polity').agg(
        count=('wikidata_id', 'count'),
        max_pid=('lowest_pid', 'max')
    ).reset_index()
    
    # Get time periods from database
    all_pids = lowest_level_counts['max_pid'].unique().tolist()
    pid_periods = {}
    for row in conn.execute(f'''
        SELECT polity_id, MIN(from_year), MAX(to_year) 
        FROM polities_periods_cliopatria 
        WHERE polity_id IN ({','.join(map(str, all_pids))})
        GROUP BY polity_id
    '''):
        pid_periods[row[0]] = (row[1], row[2])
    
    # Build metadata dataframe
    rows = []
    for _, r in lowest_level_counts.iterrows():
        pid = r['max_pid']
        if pid in pid_periods:
            start, end = pid_periods[pid]
            if start is not None and end is not None and end < MODERN_CUTOFF:
                duration = end - start
                if duration >= MIN_DURATION:
                    rows.append({
                        'polity': r['lowest_polity'],
                        'start_yr': start,
                        'collapse_yr': end,
                        'duration': duration,
                        'n_total': r['count'],
                        'polity_id': pid,
                        'type': 'long' if duration >= DURATION_THRESHOLD else 'short'
                    })
    
    df_polity_meta = pd.DataFrame(rows)
    save_checkpoint('df_polity_meta', df_polity_meta)

print(f'Non-modern polities (≥{MIN_DURATION} years): {len(df_polity_meta)}')

# ── 1c. Count individuals in windows and filter ──
df_polities = load_checkpoint('df_polities')
if df_polities is None:
    print("Counting individuals in windows...")
    polity_years = df_raw.groupby('lowest_polity')['floruit_year'].apply(list).to_dict()
    
    rows = []
    for _, p in tqdm(df_polity_meta.iterrows(), total=len(df_polity_meta), desc="Filtering"):
        years = polity_years.get(p['polity'], [])
        end = p['collapse_yr']
        
        n_crisis = sum(1 for y in years if end - CRISIS_WINDOW <= y <= end)
        n_pre = sum(1 for y in years if end - PRE_CRISIS_GAP - PRE_CRISIS_WINDOW <= y < end - PRE_CRISIS_GAP)
        
        if n_crisis + n_pre >= MIN_INDIVIDUALS:
            rows.append({**p.to_dict(), 'n_crisis': n_crisis, 'n_pre_crisis': n_pre})
    
    df_polities = pd.DataFrame(rows)
    save_checkpoint('df_polities', df_polities)

print(f"Polities meeting criteria: {len(df_polities)}")

# ── 1d. Compute polity centroids ──
df_coords = load_checkpoint('df_coords')
if df_coords is None:
    print("Computing polity centroids...")
    
    pids_needed = df_polity_meta['polity_id'].unique().tolist()
    pid_to_polity = df_polity_meta.set_index('polity_id')['polity'].to_dict()
    
    geom_query = conn.execute(f'''
        SELECT polity_id, geometry FROM (
            SELECT polity_id, geometry, ROW_NUMBER() OVER (PARTITION BY polity_id ORDER BY to_year DESC) as rn
            FROM polities_periods_cliopatria 
            WHERE polity_id IN ({','.join(map(str, pids_needed))}) AND geometry IS NOT NULL
        ) WHERE rn = 1
    ''')
    
    rows = []
    for pid, geom_str in tqdm(geom_query, desc="Parsing geometries", total=len(pids_needed)):
        polity = pid_to_polity.get(pid)
        if polity:
            try:
                geom = json.loads(geom_str)
                lat, lon = get_largest_polygon_centroid(geom)
                if lat is not None:
                    rows.append({'polity': polity, 'lat': lat, 'lon': lon})
            except:
                continue
    
    df_coords = pd.DataFrame(rows)
    save_checkpoint('df_coords', df_coords)

# Merge coords into df_polities
df_polities = df_polities.merge(df_coords, on='polity', how='left')
print(f"With coordinates: {df_polities['lat'].notna().sum()}")

# ── 1e. Find 5 nearest neighbours ACTIVE AT TIME OF COLLAPSE ──
df_neighbours = load_checkpoint('df_neighbours')
if df_neighbours is None:
    print("Finding contemporary neighbours...")
    
    # All polities with coords and time periods
    all_meta = df_polity_meta.merge(df_coords, on='polity', how='inner')
    
    coord_pnames = all_meta['polity'].tolist()
    coord_array = all_meta[['lat', 'lon']].values
    meta_array = all_meta[['start_yr', 'collapse_yr']].values
    
    rows = []
    for _, p in tqdm(df_polities.iterrows(), total=len(df_polities), desc="Computing neighbours"):
        if pd.isna(p['lat']):
            continue
        
        lat, lon = p['lat'], p['lon']
        collapse_yr = p['collapse_yr']
        pname = p['polity']
        
        # Polities active at collapse time (excluding self)
        active_mask = (meta_array[:, 0] <= collapse_yr) & (meta_array[:, 1] >= collapse_yr)
        active_mask &= np.array([n != pname for n in coord_pnames])
        
        if not active_mask.any():
            continue
        
        active_indices = np.where(active_mask)[0]
        active_coords = coord_array[active_indices]
        
        dists = np.sqrt((active_coords[:, 0] - lat)**2 +
                        ((active_coords[:, 1] - lon) * np.cos(np.radians(lat)))**2)
        
        idx5 = np.argsort(dists)[:5]
        neighbours = [coord_pnames[active_indices[i]] for i in idx5]
        
        rows.append({'polity': pname, 'neighbours': ', '.join(neighbours), 'n_neighbours': len(neighbours)})
    
    df_neighbours = pd.DataFrame(rows)
    save_checkpoint('df_neighbours', df_neighbours)

# Merge neighbours
df_polities = df_polities.drop(columns=['neighbours', 'n_neighbours'], errors='ignore')
df_polities = df_polities.merge(df_neighbours, on='polity', how='left')
df_polities['n_neighbours'] = df_polities['n_neighbours'].fillna(0).astype(int)

conn.close()

# Sort by n_total
df_polities = df_polities.sort_values('n_total', ascending=False).reset_index(drop=True)

n_short = (df_polities['type'] == 'short').sum()
n_long  = (df_polities['type'] == 'long').sum()
print(f'\nPolities: {len(df_polities)} total — {n_short} short-lived, {n_long} long-lasting')
print(f'With neighbours: {(df_polities["n_neighbours"] > 0).sum()}')

df_polities

Loading data from database...


Loaded 4,204,665 individuals
Extracting lowest-level polities...


  ✓ Saved: df_raw.pkl
Individuals: 4,204,216
Building polity metadata...


  ✓ Saved: df_polity_meta.pkl
Non-modern polities (≥26 years): 764
Counting individuals in windows...


Filtering:   0%|          | 0/764 [00:00<?, ?it/s]

...

  ✓ Saved: df_coords.pkl
With coordinates: 43
Finding contemporary neighbours...


Computing neighbours:   0%|          | 0/43 [00:00<?, ?it/s]

Computing neighbours: 100%|██████████| 43/43 [00:00<00:00, 8800.81it/s]

  ✓ Saved: df_neighbours.pkl

Polities: 43 total — 28 short-lived, 15 long-lasting
With neighbours: 43


,polity,start_yr,collapse_yr,duration,n_total,polity_id,type,n_crisis,n_pre_crisis,lat,lon,neighbours,n_neighbours
0,Kingdom of Great Britain,1709,1889,180,241744,1110,long,7513,5626,-6.437467,14.395676,"Electorate of Württemberg, Maratha Empire, Nat...",5
1,French Fifth Republic,1961,1989,28,174067,1482,short,43107,14167,29.315192,47.976589,"Kingdom of Hejaz, People's Socialist Republic ...",5
2,Kingdom of Spain,1519,1686,167,104663,1004,long,685,593,17.155060,80.602250,"Ahmadnagar Sultanate, Small Horde, Dutch Cape ...",5
3,Netherlands,1820,1870,50,87350,1244,short,1619,1218,50.734370,11.383501,"Free City of Lübeck, Austrian Empire, Kingdom ...",5
4,British Colonial Empire,1709,1894,185,37314,1106,long,2585,1430,15.042993,106.284465,"Regency of Algiers, Montenegro, Kingdom of the...",5
5,Czechoslovakia,1922,1991,69,36753,1389,short,1287,10162,59.406229,99.034049,"Estonia, Albania, Nazi Germany, Thanjavur Mara...",5
6,Austrian Empire,1806,1870,64,21127,1204,short,3862,4366,50.654419,11.790190,"Netherlands, Free City of Lübeck, Kingdom of E...",5
7,Denmark-Norway,1529,1808,279,19080,1015,long,587,415,-4.747882,104.415435,"Sultanate of Malacca, Qing Dynasty, Sultanate ...",5
8,Habsburg Monarchy,1492,1578,86,15307,981,short,135,147,20.231258,78.033311,"Ahmadnagar Sultanate, Sultanate of Cirebon, Ki...",5
9,Bourbon Kingdom of France,1820,1870,50,15059,1246,short,0,4230,50.565618,10.701557,"Kingdom of Etruria, Netherlands, Free City of ...",5


In [5]:
# ── 2. Compute occupation shares: crisis decade vs 2 decades before ──
# Crisis window: [collapse_yr - 10, collapse_yr]
# Pre-crisis window: [collapse_yr - 30, collapse_yr - 20]

df_shares = load_checkpoint('df_shares')
if df_shares is None:
    print("Loading occupation data...")
    conn = sqlite3.connect(DB)
    df_occ = pd.read_sql_query('''
        SELECT polity_name, polity_id, floruit_year, occupations
        FROM consolidated_database
        WHERE floruit_year IS NOT NULL AND occupations IS NOT NULL
    ''', conn)
    conn.close()

    # Extract lowest-level polity
    print("Extracting lowest-level polities...")
    df_occ['lowest_polity'] = df_occ['polity_name'].apply(extract_lowest_level_polity)
    df_occ = df_occ[df_occ['lowest_polity'].notna()].copy()

    # Keep only polities in our analysis set
    selected_polities = set(df_polities['polity'])
    df_occ = df_occ[df_occ['lowest_polity'].isin(selected_polities)].copy()
    print(f"Individuals in selected polities: {len(df_occ):,}")

    # Merge with polity info to get collapse_yr
    polity_info = df_polities[['polity', 'collapse_yr']].copy()
    df_occ = df_occ.merge(polity_info, left_on='lowest_polity', right_on='polity', how='inner')

    # Assign period using decade-based windows
    print("Assigning periods...")
    df_occ['period'] = np.select(
        [
            (df_occ['floruit_year'] >= df_occ['collapse_yr'] - CRISIS_WINDOW) & 
            (df_occ['floruit_year'] <= df_occ['collapse_yr']),
            (df_occ['floruit_year'] >= df_occ['collapse_yr'] - PRE_CRISIS_GAP - PRE_CRISIS_WINDOW) & 
            (df_occ['floruit_year'] < df_occ['collapse_yr'] - PRE_CRISIS_GAP)
        ],
        ['crisis', 'pre_crisis'],
        default=None
    )

    # Keep only rows in analysis windows
    df_occ = df_occ[df_occ['period'].notna()].copy()
    print(f"Individuals in analysis windows: {len(df_occ):,}")

    # Explode occupations
    print("Exploding occupations...")
    df_occ['occupation'] = df_occ['occupations'].str.split('; ')
    df_occ = df_occ.explode('occupation')
    df_occ['occupation'] = df_occ['occupation'].str.strip()
    df_occ = df_occ[df_occ['occupation'] != ''].copy()
    print(f"Individual-occupation pairs: {len(df_occ):,}")

    # Count occupations per polity (total across both periods) to filter >= 10
    print("Filtering occupations (≥10 per polity)...")
    occ_counts_total = df_occ.groupby(['lowest_polity', 'occupation']).size().reset_index(name='total_count')
    occ_counts_total = occ_counts_total[occ_counts_total['total_count'] >= 10]
    
    # Merge to filter
    df_occ = df_occ.merge(occ_counts_total[['lowest_polity', 'occupation']], 
                          on=['lowest_polity', 'occupation'], how='inner')
    print(f"After filtering: {len(df_occ):,} pairs")

    # Count by polity, occupation, period
    print("Computing shares...")
    counts = df_occ.groupby(['lowest_polity', 'occupation', 'period']).size().reset_index(name='count')
    totals = df_occ.groupby(['lowest_polity', 'period']).size().reset_index(name='total')
    counts = counts.merge(totals, on=['lowest_polity', 'period'])
    counts['share'] = counts['count'] / counts['total']

    # Pivot to wide format
    df_shares = counts.pivot_table(
        index=['lowest_polity', 'occupation'],
        columns='period',
        values=['count', 'share', 'total'],
        fill_value=0
    ).reset_index()

    # Flatten column names
    df_shares.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in df_shares.columns]

    # Compute changes
    df_shares['share_change'] = df_shares['share_crisis'] - df_shares['share_pre_crisis']
    df_shares['share_pct_change'] = ((df_shares['share_crisis'] - df_shares['share_pre_crisis']) / 
                                      df_shares['share_pre_crisis'].replace(0, np.nan) * 100)

    # Rename columns
    df_shares = df_shares.rename(columns={
        'lowest_polity': 'polity',
        'count_pre_crisis': 'n_pre_crisis',
        'count_crisis': 'n_crisis',
        'total_pre_crisis': 'total_pre_crisis',
        'total_crisis': 'total_crisis'
    })

    # Sort by absolute share change
    df_shares = df_shares.sort_values('share_change', key=abs, ascending=False).reset_index(drop=True)
    
    save_checkpoint('df_shares', df_shares)

print(f"\nOccupation shares:")
print(f"  Polities: {df_shares['polity'].nunique()}")
print(f"  Occupations: {df_shares['occupation'].nunique()}")
print(f"  Total rows: {len(df_shares):,}")

df_shares

Loading occupation data...


Extracting lowest-level polities...


Individuals in selected polities: 871,545
Assigning periods...
Individuals in analysis windows: 121,709
Exploding occupations...


Individual-occupation pairs: 227,232
Filtering occupations (≥10 per polity)...
After filtering: 203,822 pairs
Computing shares...
  ✓ Saved: df_shares.pkl

Occupation shares:
  Polities: 43
  Occupations: 768
  Total rows: 2,198


,polity,occupation,n_crisis,n_pre_crisis,share_crisis,share_pre_crisis,total_crisis,total_pre_crisis,share_change,share_pct_change
0,House of Habsburg,painter,16.0,2.0,0.800000,0.250000,20.0,8.0,0.550000,220.000000
1,House of Habsburg,politician,4.0,6.0,0.200000,0.750000,20.0,8.0,-0.550000,-73.333333
2,Spanish Empire,politician,75.0,3.0,0.462963,0.103448,162.0,29.0,0.359515,347.530864
3,Yugoslavia,partisan,0.0,107.0,0.000000,0.277922,0.0,385.0,-0.277922,-100.000000
4,Habsburg Monarchy,painter,1.0,33.0,0.006803,0.272727,147.0,121.0,-0.265925,-97.505669
...,...,...,...,...,...,...,...,...,...,...
2193,Czechoslovakia,reporter,1.0,9.0,0.000401,0.000399,2493.0,22563.0,0.000002,0.561572
2194,Czechoslovakia,saxophonist,1.0,9.0,0.000401,0.000399,2493.0,22563.0,0.000002,0.561572
2195,Austrian Empire,wood carver,7.0,8.0,0.001026,0.001028,6821.0,7785.0,-0.000001,-0.133778
2196,French Fifth Republic,performing artist,23.0,7.0,0.000286,0.000285,80430.0,24580.0,0.000001,0.413847


In [6]:
# ── 3. Wilcoxon Signed-Rank Test ──
# Test if occupation share changes are systematically different from zero across polities

from scipy.stats import wilcoxon

MIN_POLITIES = 8  # minimum polities needed for Wilcoxon test

# Group share changes by occupation
occ_groups = df_shares.groupby('occupation')['share_change'].apply(list).to_dict()

results = []
for occ, deltas in occ_groups.items():
    deltas = [d for d in deltas if not np.isnan(d)]
    
    if len(deltas) < MIN_POLITIES:
        continue
    
    arr = np.array(deltas)
    
    # Skip if all zeros
    if np.all(arr == 0):
        continue
    
    try:
        stat, pval = wilcoxon(arr)
    except ValueError:
        continue
    
    mean_d = np.mean(arr)
    median_d = np.median(arr)
    
    results.append({
        'occupation': occ,
        'direction': 'RISE' if mean_d > 0 else 'FALL',
        'n_polities': len(deltas),
        'mean_delta': mean_d,
        'median_delta': median_d,
        'pvalue': pval,
        'consistency': np.mean([(d > 0) == (mean_d > 0) for d in deltas]) * 100,
    })

# Sort by p-value
results.sort(key=lambda x: x['pvalue'])

# Convert to DataFrame
df_wilcoxon = pd.DataFrame(results)

n_sig = (df_wilcoxon['pvalue'] < 0.05).sum() if len(df_wilcoxon) > 0 else 0
print(f"Wilcoxon signed-rank test results:")
print(f"  Occupations tested: {len(df_wilcoxon)}")
print(f"  Significant (p < 0.05): {n_sig}")

# Show top 30
print(f"\nTop 30 collapse-associated occupations:")
df_wilcoxon.head(30)

Wilcoxon signed-rank test results:
  Occupations tested: 52
  Significant (p < 0.05): 3

Top 30 collapse-associated occupations:


,occupation,direction,n_polities,mean_delta,median_delta,pvalue,consistency
0,botanical collector,FALL,8,-0.001444,-0.000969,0.007812,100.000000
1,painter,FALL,31,-0.015652,-0.008680,0.008061,74.193548
2,military personnel,FALL,30,-0.024496,-0.007368,0.019661,70.000000
3,university teacher,FALL,20,-0.006975,-0.005950,0.069580,55.000000
4,physician,FALL,24,-0.007210,-0.004270,0.073793,70.833333
5,civil servant,FALL,13,-0.003728,-0.002328,0.080322,84.615385
6,judge,FALL,13,-0.008690,-0.002480,0.080322,69.230769
7,historian,FALL,18,-0.003954,-0.006136,0.089767,61.111111
8,draftsperson,FALL,11,-0.001985,-0.003197,0.147461,72.727273
9,linguist,FALL,9,-0.001286,-0.001702,0.164062,77.777778


In [7]:
# ── 4. Difference-in-Differences: Collapsed vs Neighbors ──
# For each collapsed polity, compare its occupation share change against the AVERAGE of its 5 neighbors
# DiD = (collapsed share change) - (average of neighbors' share changes)

print("Computing difference-in-differences...")
conn = sqlite3.connect(DB)

# Load occupation data
df_occ_all = pd.read_sql_query('''
    SELECT polity_name, polity_id, floruit_year, occupations
    FROM consolidated_database
    WHERE floruit_year IS NOT NULL AND occupations IS NOT NULL
''', conn)
conn.close()

# Extract lowest-level polity
df_occ_all['lowest_polity'] = df_occ_all['polity_name'].apply(extract_lowest_level_polity)
df_occ_all = df_occ_all[df_occ_all['lowest_polity'].notna()].copy()

def compute_share_change_for_polity(df_polity, collapse_yr):
    """Compute share change for a single polity given a collapse year reference."""
    # Assign periods
    df_polity = df_polity.copy()
    df_polity['period'] = np.select(
        [
            (df_polity['floruit_year'] >= collapse_yr - CRISIS_WINDOW) & 
            (df_polity['floruit_year'] <= collapse_yr),
            (df_polity['floruit_year'] >= collapse_yr - PRE_CRISIS_GAP - PRE_CRISIS_WINDOW) & 
            (df_polity['floruit_year'] < collapse_yr - PRE_CRISIS_GAP)
        ],
        ['crisis', 'pre_crisis'],
        default=None
    )
    
    df_polity = df_polity[df_polity['period'].notna()].copy()
    if len(df_polity) == 0:
        return None
    
    # Explode occupations
    df_polity['occupation'] = df_polity['occupations'].str.split('; ')
    df_polity = df_polity.explode('occupation')
    df_polity['occupation'] = df_polity['occupation'].str.strip()
    df_polity = df_polity[df_polity['occupation'] != ''].copy()
    
    if len(df_polity) == 0:
        return None
    
    # Compute shares
    counts = df_polity.groupby(['occupation', 'period']).size().reset_index(name='count')
    totals = df_polity.groupby('period').size().reset_index(name='total')
    counts = counts.merge(totals, on='period')
    counts['share'] = counts['count'] / counts['total']
    
    # Pivot
    shares = counts.pivot(index='occupation', columns='period', values='share').reset_index()
    shares.columns.name = None
    
    if 'crisis' not in shares.columns or 'pre_crisis' not in shares.columns:
        return None
    
    shares['share_change'] = shares['crisis'].fillna(0) - shares['pre_crisis'].fillna(0)
    return shares[['occupation', 'share_change']].set_index('occupation')['share_change'].to_dict()

# Compute DiD for each collapsed polity
did_results = []

for _, row in tqdm(df_polities.iterrows(), total=len(df_polities), desc="Computing DiD"):
    polity = row['polity']
    collapse_yr = row['collapse_yr']
    neighbours_str = row['neighbours']
    
    if pd.isna(neighbours_str) or not neighbours_str:
        continue
    
    neighbours = [n.strip() for n in neighbours_str.split(',')]
    
    # Get collapsed polity's share changes
    collapsed_shares = df_shares[df_shares['polity'] == polity].set_index('occupation')['share_change'].to_dict()
    
    if not collapsed_shares:
        continue
    
    # Compute share change for EACH neighbor separately
    neighbor_changes = {}  # occupation -> list of changes from each neighbor
    
    for neighbor in neighbours:
        df_neighbor = df_occ_all[df_occ_all['lowest_polity'] == neighbor].copy()
        if len(df_neighbor) == 0:
            continue
        
        neighbor_share_changes = compute_share_change_for_polity(df_neighbor, collapse_yr)
        if neighbor_share_changes is None:
            continue
        
        for occ, change in neighbor_share_changes.items():
            if occ not in neighbor_changes:
                neighbor_changes[occ] = []
            neighbor_changes[occ].append(change)
    
    # Compute DiD: collapsed - average(neighbors)
    for occ, collapsed_change in collapsed_shares.items():
        if occ not in neighbor_changes or len(neighbor_changes[occ]) == 0:
            continue
        
        avg_neighbor_change = np.mean(neighbor_changes[occ])
        did = collapsed_change - avg_neighbor_change
        
        did_results.append({
            'polity': polity,
            'occupation': occ,
            'collapsed_share_change': collapsed_change,
            'avg_neighbor_change': avg_neighbor_change,
            'n_neighbors': len(neighbor_changes[occ]),
            'did': did
        })

# Combine all results
df_did = pd.DataFrame(did_results)
print(f"\nDiD results: {len(df_did):,} polity-occupation pairs")
print(f"Average neighbors per comparison: {df_did['n_neighbors'].mean():.1f}")

# Save checkpoint
save_checkpoint('df_did', df_did)

df_did

Computing difference-in-differences...


Computing DiD:   0%|          | 0/43 [00:00<?, ?it/s]

Computing DiD:   5%|▍         | 2/43 [00:00<00:03, 11.27it/s]

Computing DiD:   9%|▉         | 4/43 [00:00<00:03, 10.86it/s]

Computing DiD:  14%|█▍        | 6/43 [00:00<00:03, 12.29it/s]

...

Computing DiD:  93%|█████████▎| 40/43 [00:03<00:00, 12.87it/s]

Computing DiD:  98%|█████████▊| 42/43 [00:03<00:00, 13.02it/s]

Computing DiD: 100%|██████████| 43/43 [00:03<00:00, 12.46it/s]


DiD results: 800 polity-occupation pairs
Average neighbors per comparison: 1.7
  ✓ Saved: df_did.pkl


,polity,occupation,collapsed_share_change,avg_neighbor_change,n_neighbors,did
0,Kingdom of Great Britain,politician,-0.016967,0.125000,1,-0.141967
1,Kingdom of Great Britain,military personnel,-0.011089,0.125000,1,-0.136089
2,Kingdom of Great Britain,soldier,-0.002336,-0.333333,1,0.330998
3,Kingdom of Great Britain,stockbroker,0.000893,0.125000,1,-0.124107
4,Kingdom of Great Britain,teacher,0.000822,0.125000,1,-0.124178
...,...,...,...,...,...,...
795,Portuguese Empire,painter,-0.161310,0.065806,1,-0.227115
796,Portuguese Empire,writer,0.016071,-0.001302,1,0.017372
797,Duchy of Nassau,politician,-0.100000,0.059925,4,-0.159925
798,Duchy of Nassau,painter,0.100000,-0.028353,3,0.128353


In [8]:
# ── 5. Wilcoxon Test on DiD Estimates ──
# Test if DiD (collapse-specific signal) is systematically different from zero

MIN_POLITIES = 8

# Group DiD by occupation
did_groups = df_did.groupby('occupation')['did'].apply(list).to_dict()

results_did = []
for occ, deltas in did_groups.items():
    deltas = [d for d in deltas if not np.isnan(d)]
    
    if len(deltas) < MIN_POLITIES:
        continue
    
    arr = np.array(deltas)
    
    if np.all(arr == 0):
        continue
    
    try:
        stat, pval = wilcoxon(arr)
    except ValueError:
        continue
    
    mean_d = np.mean(arr)
    median_d = np.median(arr)
    
    results_did.append({
        'occupation': occ,
        'direction': 'RISE' if mean_d > 0 else 'FALL',
        'n_polities': len(deltas),
        'mean_did': mean_d,
        'median_did': median_d,
        'pvalue': pval,
        'consistency': np.mean([(d > 0) == (mean_d > 0) for d in deltas]) * 100,
    })

# Sort by p-value
results_did.sort(key=lambda x: x['pvalue'])

# Convert to DataFrame
df_wilcoxon_did = pd.DataFrame(results_did)

# Compare with raw results
if len(df_wilcoxon) > 0:
    raw_rank = {r: i+1 for i, r in enumerate(df_wilcoxon['occupation'])}
    df_wilcoxon_did['raw_rank'] = df_wilcoxon_did['occupation'].map(raw_rank)

n_sig = (df_wilcoxon_did['pvalue'] < 0.05).sum() if len(df_wilcoxon_did) > 0 else 0
print(f"DiD Wilcoxon signed-rank test results:")
print(f"  Occupations tested: {len(df_wilcoxon_did)}")
print(f"  Significant (p < 0.05): {n_sig}")

# Show top 30
print(f"\nTop 30 collapse-specific signals (DiD-controlled):")
df_wilcoxon_did.head(30)

DiD Wilcoxon signed-rank test results:
  Occupations tested: 22
  Significant (p < 0.05): 5

Top 30 collapse-specific signals (DiD-controlled):


,occupation,direction,n_polities,mean_did,median_did,pvalue,consistency,raw_rank
0,architect,FALL,8,-0.014741,-0.011721,0.007812,100.000000,12
1,painter,FALL,18,-0.047561,-0.031500,0.010406,77.777778,2
2,writer,FALL,23,-0.051606,-0.015294,0.017873,65.217391,28
3,poet,FALL,21,-0.055477,-0.021731,0.035056,71.428571,31
4,civil servant,FALL,9,-0.015187,-0.007642,0.039062,88.888889,6
5,military personnel,FALL,18,-0.043042,-0.025028,0.053856,66.666667,3
6,engineer,FALL,9,-0.007883,-0.009512,0.097656,77.777778,23
7,journalist,FALL,15,-0.015659,-0.014413,0.252380,66.666667,35
8,lawyer,FALL,12,-0.012305,-0.005204,0.266113,75.000000,27
9,physician,FALL,13,-0.004072,-0.007445,0.305420,69.230769,5
